In [0]:
CUSTOMER_FEATURES_TABLE = "workspace.marketing_campaign.gold_customer_features"
SUMMARY_TABLE = "workspace.marketing_campaign.gold_campaign_summary"

features_df = spark.table(CUSTOMER_FEATURES_TABLE)

print(f"Customer features table: {CUSTOMER_FEATURES_TABLE}")
print(f"Summary table: {SUMMARY_TABLE}")
print(f"Feature rows: {features_df.count()}")

In [0]:
from pyspark.sql.functions import col, when

summary_base_df = (
    features_df
    .withColumn(
        "age_band",
        when(col("customer_age") < 30, "Under 30")
        .when(col("customer_age") < 45, "30-44")
        .when(col("customer_age") < 60, "45-59")
        .otherwise("60+")
    )
    .withColumn(
        "income_band",
        when(col("income") < 30000, "Under 30k")
        .when(col("income") < 60000, "30k-59k")
        .when(col("income") < 90000, "60k-89k")
        .otherwise("90k+")
    )
)

In [0]:
from pyspark.sql.functions import count, avg, sum as spark_sum, round, lit, col

def create_segment_summary(segment_column):
    return (
        summary_base_df
        .groupBy(segment_column)
        .agg(
            count("*").alias("customer_count"),
            spark_sum("response").alias("responded_count"),
            round(avg("response"), 4).alias("response_rate"),
            round(avg("income"), 2).alias("avg_income"),
            round(avg("total_spend"), 2).alias("avg_total_spend"),
            round(avg("total_purchases"), 2).alias("avg_total_purchases")
        )
        .withColumn("segment_type", lit(segment_column))
        .withColumnRenamed(segment_column, "segment_value")
        .withColumn("segment_value", col("segment_value").cast("string"))
    )

In [0]:
from pyspark.sql.functions import lit

education_summary_df = create_segment_summary("education")
marital_summary_df = create_segment_summary("marital_status")
age_summary_df = create_segment_summary("age_band")
income_summary_df = create_segment_summary("income_band")
children_summary_df = create_segment_summary("has_children")
previous_campaign_summary_df = create_segment_summary("accepted_previous_campaign")

In [0]:
summary_df = (
    education_summary_df
    .unionByName(marital_summary_df)
    .unionByName(age_summary_df)
    .unionByName(income_summary_df)
    .unionByName(children_summary_df)
    .unionByName(previous_campaign_summary_df)
)

display(summary_df)

In [0]:
(
    summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SUMMARY_TABLE)
)

In [0]:
summary_check_df = spark.table(SUMMARY_TABLE)

print(f"Rows saved: {summary_check_df.count()}")
print(f"Columns saved: {len(summary_check_df.columns)}")

display(summary_check_df)